In [1]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V10
=================================================

KEY PARAMETERS:
- RNA (Visium): ~55 μm pixel size, hexagonal grid, 6 neighbors
- MSI: 60 μm pixel size, Cartesian grid, 8 neighbors
- RNA needs 180° rotation to align with MSI

APPROACH:
This version uses Cross-Modal Transformers exclusively for similarity computation.
The transformer learns to align RNA and MSI modalities in a shared embedding space
using attention mechanisms and learned positional encodings.

Changes from V9:
- Removed coordinate-based similarity computation
- Removed descriptor-based similarity computation  
- Removed GNN models entirely
- Combined score now uses only transformer-based metrics (100%)

Author: V10 - Transformer-Only Similarity
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
from typing import Dict, Tuple, Optional
import pandas as pd
import os
import warnings
from dataclasses import dataclass
import pickle
import math

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

RNA_PIXEL_SIZE = 55  # μm (Visium)
MSI_PIXEL_SIZE = 60  # μm

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = ['Thy1', 'App', 'Apoe', 'Mapt']


# =============================================================================
# COORDINATE TRANSFORMATION
# =============================================================================

def rotate_180(coords: np.ndarray) -> np.ndarray:
    """Rotate coordinates 180° around centroid"""
    center = coords.mean(axis=0)
    return 2 * center - coords


def align_rna_to_msi(rna_coords: np.ndarray, msi_coords: np.ndarray) -> np.ndarray:
    """Align RNA coordinates to MSI coordinate system with 180° rotation and scaling"""
    rotated = rotate_180(rna_coords)
    
    rna_min, rna_max = rotated.min(axis=0), rotated.max(axis=0)
    msi_min, msi_max = msi_coords.min(axis=0), msi_coords.max(axis=0)
    
    rna_range = rna_max - rna_min
    msi_range = msi_max - msi_min
    scale = msi_range / (rna_range + 1e-8)
    
    return (rotated - rna_min) * scale + msi_min


# =============================================================================
# LEARNED POSITIONAL ENCODING
# =============================================================================

class LearnedPositionalEncoding2D(nn.Module):
    """
    Learned 2D positional encoding for spatial coordinates.
    
    Combines learnable embeddings with sinusoidal features for both
    absolute position and relative spatial relationships.
    """
    
    def __init__(self, d_model: int, max_spatial_size: int = 1000, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.max_spatial_size = max_spatial_size
        
        # Learnable position embeddings (grid-based)
        self.x_embedding = nn.Embedding(max_spatial_size, d_model // 2)
        self.y_embedding = nn.Embedding(max_spatial_size, d_model // 2)
        
        # Continuous coordinate projection (for non-grid positions)
        self.coord_projection = nn.Sequential(
            nn.Linear(2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
        # Learnable scale parameters for combining embeddings
        self.alpha = nn.Parameter(torch.ones(1) * 0.5)
        self.beta = nn.Parameter(torch.ones(1) * 0.5)
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        
    def _get_sinusoidal_encoding(self, coords: torch.Tensor) -> torch.Tensor:
        """Generate sinusoidal positional encoding from continuous coordinates"""
        batch_size, n_points, _ = coords.shape
        device = coords.device
        
        # Scale coordinates to [0, 1]
        coords_norm = coords.clone()
        for b in range(batch_size):
            c_min = coords[b].min(dim=0)[0]
            c_max = coords[b].max(dim=0)[0]
            coords_norm[b] = (coords[b] - c_min) / (c_max - c_min + 1e-8)
        
        # Generate frequency bands
        d_half = self.d_model // 4
        freq_bands = torch.exp(
            torch.arange(0, d_half, dtype=torch.float32, device=device) *
            (-math.log(10000.0) / d_half)
        )
        
        # Apply sinusoidal encoding to x and y separately
        x_enc = coords_norm[..., 0:1] * freq_bands  # [B, N, d_half]
        y_enc = coords_norm[..., 1:2] * freq_bands  # [B, N, d_half]
        
        pe = torch.cat([
            torch.sin(x_enc), torch.cos(x_enc),
            torch.sin(y_enc), torch.cos(y_enc)
        ], dim=-1)
        
        return pe
    
    def forward(self, x: torch.Tensor, coords: torch.Tensor) -> torch.Tensor:
        """
        Add positional encoding to input features.
        
        Args:
            x: Input features [batch_size, n_points, d_model]
            coords: Spatial coordinates [batch_size, n_points, 2]
        
        Returns:
            Position-encoded features [batch_size, n_points, d_model]
        """
        # Handle unbatched input
        if x.dim() == 2:
            x = x.unsqueeze(0)
            coords = coords.unsqueeze(0)
            squeeze_output = True
        else:
            squeeze_output = False
        
        batch_size, n_points, _ = x.shape
        device = x.device
        
        # Normalize coordinates to grid indices
        coords_norm = coords.clone()
        for b in range(batch_size):
            c_min = coords[b].min(dim=0)[0]
            c_max = coords[b].max(dim=0)[0]
            coords_norm[b] = (coords[b] - c_min) / (c_max - c_min + 1e-8)
        
        # Get grid indices (clamped to valid range)
        x_idx = (coords_norm[..., 0] * (self.max_spatial_size - 1)).long().clamp(0, self.max_spatial_size - 1)
        y_idx = (coords_norm[..., 1] * (self.max_spatial_size - 1)).long().clamp(0, self.max_spatial_size - 1)
        
        # Learnable grid embeddings
        x_emb = self.x_embedding(x_idx)  # [B, N, d_model//2]
        y_emb = self.y_embedding(y_idx)  # [B, N, d_model//2]
        grid_pe = torch.cat([x_emb, y_emb], dim=-1)  # [B, N, d_model]
        
        # Continuous coordinate projection
        coord_pe = self.coord_projection(coords)  # [B, N, d_model]
        
        # Sinusoidal encoding
        sin_pe = self._get_sinusoidal_encoding(coords)  # [B, N, d_model]
        
        # Combine all positional encodings with learnable weights
        alpha = torch.sigmoid(self.alpha)
        beta = torch.sigmoid(self.beta)
        
        combined_pe = alpha * grid_pe + beta * coord_pe + (1 - alpha - beta).clamp(min=0) * sin_pe
        
        # Add to input and normalize
        output = self.norm(x + self.dropout(combined_pe))
        
        if squeeze_output:
            output = output.squeeze(0)
        
        return output


# =============================================================================
# CROSS-MODAL TRANSFORMER
# =============================================================================

class CrossModalAttention(nn.Module):
    """
    Cross-modal attention layer for aligning RNA and MSI modalities.
    
    Uses multi-head attention where queries come from one modality
    and keys/values come from the other modality.
    """
    
    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Cross-modal attention.
        
        Args:
            query: Query tensor [batch, n_query, d_model]
            key: Key tensor [batch, n_key, d_model]
            value: Value tensor [batch, n_key, d_model]
            mask: Optional attention mask
        
        Returns:
            output: Attended features [batch, n_query, d_model]
            attention: Attention weights [batch, n_heads, n_query, n_key]
        """
        batch_size = query.size(0)
        
        # Linear projections
        Q = self.w_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)
        
        # Apply attention to values
        context = torch.matmul(attention, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        output = self.w_o(context)
        
        return output, attention


class CrossModalTransformerLayer(nn.Module):
    """
    Single layer of the Cross-Modal Transformer.
    
    Contains:
    - Self-attention within each modality
    - Cross-attention between modalities
    - Feed-forward network
    """
    
    def __init__(self, d_model: int, n_heads: int = 8, d_ff: int = 512, dropout: float = 0.1):
        super().__init__()
        
        # Self-attention
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.self_norm = nn.LayerNorm(d_model)
        
        # Cross-attention
        self.cross_attn = CrossModalAttention(d_model, n_heads, dropout)
        self.cross_norm = nn.LayerNorm(d_model)
        
        # Feed-forward
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.ff_norm = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(
        self,
        x: torch.Tensor,
        context: torch.Tensor,
        x_coords: Optional[torch.Tensor] = None,
        context_coords: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through transformer layer.
        
        Args:
            x: Input features from source modality [batch, n_source, d_model]
            context: Features from target modality [batch, n_target, d_model]
            x_coords: Optional spatial coordinates for source
            context_coords: Optional spatial coordinates for target
        
        Returns:
            output: Transformed features [batch, n_source, d_model]
            cross_attention: Cross-modal attention weights
        """
        # Self-attention
        self_out, _ = self.self_attn(x, x, x)
        x = self.self_norm(x + self.dropout(self_out))
        
        # Cross-attention
        cross_out, cross_attn = self.cross_attn(x, context, context)
        x = self.cross_norm(x + self.dropout(cross_out))
        
        # Feed-forward
        ff_out = self.ff(x)
        x = self.ff_norm(x + ff_out)
        
        return x, cross_attn


class CrossModalTransformer(nn.Module):
    """
    Cross-Modal Transformer for aligning RNA and MSI spatial patterns.
    
    Uses learned positional encodings and bidirectional cross-attention
    to learn a shared embedding space between modalities.
    """
    
    def __init__(
        self,
        d_model: int = 128,
        n_heads: int = 8,
        n_layers: int = 4,
        d_ff: int = 512,
        dropout: float = 0.1,
        max_spatial_size: int = 1000
    ):
        super().__init__()
        
        self.d_model = d_model
        
        # Modality-specific input projections (will be created dynamically)
        self.rna_projections = nn.ModuleDict()
        self.msi_projections = nn.ModuleDict()
        
        # Learned positional encodings for each modality
        self.rna_pos_enc = LearnedPositionalEncoding2D(d_model, max_spatial_size, dropout)
        self.msi_pos_enc = LearnedPositionalEncoding2D(d_model, max_spatial_size, dropout)
        
        # Modality embeddings (learnable tokens to distinguish RNA vs MSI)
        self.rna_modality_emb = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.msi_modality_emb = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        # Transformer layers (RNA → MSI direction)
        self.rna_to_msi_layers = nn.ModuleList([
            CrossModalTransformerLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # Transformer layers (MSI → RNA direction)
        self.msi_to_rna_layers = nn.ModuleList([
            CrossModalTransformerLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # Global pooling attention
        self.pool_attention = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1)
        )
        
        # Output projection to shared embedding space
        self.output_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
    def _get_projection(self, dim: int, modality: str) -> nn.Module:
        """Get or create input projection for given dimension"""
        key = str(dim)
        projections = self.rna_projections if modality == 'rna' else self.msi_projections
        
        if key not in projections:
            device = next(self.parameters()).device
            projections[key] = nn.Sequential(
                nn.Linear(dim, self.d_model),
                nn.LayerNorm(self.d_model),
                nn.GELU()
            ).to(device)
        
        return projections[key]
    
    def _global_pool(self, x: torch.Tensor) -> torch.Tensor:
        """Attention-based global pooling"""
        attn_weights = F.softmax(self.pool_attention(x).squeeze(-1), dim=-1)
        return (attn_weights.unsqueeze(-1) * x).sum(dim=-2)
    
    def forward(
        self,
        rna_features: torch.Tensor,
        rna_coords: torch.Tensor,
        msi_features: torch.Tensor,
        msi_coords: torch.Tensor,
        return_squeezed: bool = True
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass through cross-modal transformer.
        
        Args:
            rna_features: RNA features [batch, n_rna, d_rna] or [n_rna, d_rna]
            rna_coords: RNA spatial coordinates [batch, n_rna, 2] or [n_rna, 2]
            msi_features: MSI features [batch, n_msi, d_msi] or [n_msi, d_msi]
            msi_coords: MSI spatial coordinates [batch, n_msi, 2] or [n_msi, 2]
            return_squeezed: If True and input was unbatched, squeeze output
        
        Returns:
            Dictionary containing:
            - rna_embedding: Global RNA embedding
            - msi_embedding: Global MSI embedding
            - rna_node_embeddings: Per-node RNA embeddings
            - msi_node_embeddings: Per-node MSI embeddings
            - cross_attention_rna_to_msi: Attention weights RNA→MSI
            - cross_attention_msi_to_rna: Attention weights MSI→RNA
            - similarity_score: Cross-modal similarity score
        """
        # Handle unbatched input
        input_was_unbatched = rna_features.dim() == 2
        if input_was_unbatched:
            rna_features = rna_features.unsqueeze(0)
            rna_coords = rna_coords.unsqueeze(0)
            msi_features = msi_features.unsqueeze(0)
            msi_coords = msi_coords.unsqueeze(0)
        
        batch_size = rna_features.size(0)
        
        # Project inputs to common dimension
        rna_proj = self._get_projection(rna_features.size(-1), 'rna')
        msi_proj = self._get_projection(msi_features.size(-1), 'msi')
        
        rna_h = rna_proj(rna_features)
        msi_h = msi_proj(msi_features)
        
        # Add positional encodings
        rna_h = self.rna_pos_enc(rna_h, rna_coords)
        msi_h = self.msi_pos_enc(msi_h, msi_coords)
        
        # Add modality embeddings
        rna_h = rna_h + self.rna_modality_emb.expand(batch_size, rna_h.size(1), -1)
        msi_h = msi_h + self.msi_modality_emb.expand(batch_size, msi_h.size(1), -1)
        
        # Store attention weights
        rna_to_msi_attns = []
        msi_to_rna_attns = []
        
        # Process through transformer layers (bidirectional)
        rna_out = rna_h
        msi_out = msi_h
        
        for rna_layer, msi_layer in zip(self.rna_to_msi_layers, self.msi_to_rna_layers):
            # RNA attends to MSI
            rna_out, rna_attn = rna_layer(rna_out, msi_out, rna_coords, msi_coords)
            rna_to_msi_attns.append(rna_attn)
            
            # MSI attends to RNA
            msi_out, msi_attn = msi_layer(msi_out, rna_out, msi_coords, rna_coords)
            msi_to_rna_attns.append(msi_attn)
        
        # Global pooling
        rna_global = self._global_pool(rna_out)
        msi_global = self._global_pool(msi_out)
        
        # Combined embedding
        combined = torch.cat([rna_global, msi_global], dim=-1)
        shared_embedding = self.output_proj(combined)
        
        # Compute similarity score
        rna_norm = F.normalize(rna_global, p=2, dim=-1)
        msi_norm = F.normalize(msi_global, p=2, dim=-1)
        similarity = (rna_norm * msi_norm).sum(dim=-1)
        
        outputs = {
            'rna_embedding': rna_global,
            'msi_embedding': msi_global,
            'shared_embedding': shared_embedding,
            'rna_node_embeddings': rna_out,
            'msi_node_embeddings': msi_out,
            'cross_attention_rna_to_msi': torch.stack(rna_to_msi_attns),
            'cross_attention_msi_to_rna': torch.stack(msi_to_rna_attns),
            'similarity_score': similarity
        }
        
        # Only squeeze if input was unbatched AND caller wants squeezed output
        if input_was_unbatched and return_squeezed:
            for key in ['rna_node_embeddings', 'msi_node_embeddings', 
                        'cross_attention_rna_to_msi', 'cross_attention_msi_to_rna']:
                if key in outputs and isinstance(outputs[key], torch.Tensor):
                    if outputs[key].dim() > 0 and outputs[key].size(0) == 1:
                        outputs[key] = outputs[key].squeeze(0)
        
        return outputs


class CrossModalContrastiveLoss(nn.Module):
    """
    Contrastive loss for training the Cross-Modal Transformer.
    
    Encourages matching RNA-MSI pairs to have similar embeddings
    while pushing non-matching pairs apart.
    """
    
    def __init__(self, temperature: float = 0.07, margin: float = 0.5):
        super().__init__()
        self.temperature = temperature
        self.margin = margin
        
    def forward(
        self,
        rna_embeddings: torch.Tensor,
        msi_embeddings: torch.Tensor,
        labels: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Compute contrastive loss.
        
        Args:
            rna_embeddings: RNA embeddings [batch, d_model]
            msi_embeddings: MSI embeddings [batch, d_model]
            labels: Optional matching labels (1 for match, 0 for non-match)
        
        Returns:
            Contrastive loss value
        """
        # Ensure 2D tensors
        if rna_embeddings.dim() == 1:
            rna_embeddings = rna_embeddings.unsqueeze(0)
        if msi_embeddings.dim() == 1:
            msi_embeddings = msi_embeddings.unsqueeze(0)
        
        batch_size = rna_embeddings.size(0)
        
        # Normalize embeddings
        rna_norm = F.normalize(rna_embeddings, p=2, dim=-1)
        msi_norm = F.normalize(msi_embeddings, p=2, dim=-1)
        
        # For single sample, use cosine similarity loss directly
        if batch_size == 1:
            # Maximize similarity for matching pairs
            similarity = (rna_norm * msi_norm).sum(dim=-1)
            loss = 1 - similarity.mean()
            return loss
        
        # For batch, compute similarity matrix
        similarity = torch.matmul(rna_norm, msi_norm.T) / self.temperature
        
        if labels is None:
            # Assume diagonal elements are positive pairs
            labels = torch.eye(batch_size, device=rna_embeddings.device)
        
        # InfoNCE loss
        log_softmax = F.log_softmax(similarity, dim=-1)
        loss = -(labels * log_softmax).sum(dim=-1).mean()
        
        return loss


# =============================================================================
# SPATIAL SIGNATURE (EXTENDED)
# =============================================================================

@dataclass
class SpatialSignature:
    """Spatial signature with transformer embeddings"""
    sample_id: str
    feature_name: str
    feature_type: str
    
    # Cross-Modal Transformer embeddings
    transformer_embedding: Optional[np.ndarray] = None
    
    # Raw data
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None
    
    # Aligned coordinates
    aligned_coordinates: Optional[np.ndarray] = None


# =============================================================================
# SIMILARITY COMPUTATION
# =============================================================================

def compute_transformer_similarity(gene_sig: SpatialSignature, mz_sig: SpatialSignature) -> dict:
    """Compute similarity using Cross-Modal Transformer embeddings"""
    
    if gene_sig.transformer_embedding is None or mz_sig.transformer_embedding is None:
        return {
            'transformer_cosine': 0,
            'transformer_l2_distance': 1.0
        }
    
    g_emb = gene_sig.transformer_embedding.flatten()
    m_emb = mz_sig.transformer_embedding.flatten()
    
    # Cosine similarity
    cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
    
    # L2 distance (normalized)
    l2_dist = np.linalg.norm(g_emb - m_emb)
    l2_sim = 1 / (1 + l2_dist)
    
    return {
        'transformer_cosine': cosine,
        'transformer_l2_similarity': l2_sim
    }


def compute_combined_score(trans_sim: dict) -> float:
    """
    Combined score using transformer-based metrics only.
    
    The Cross-Modal Transformer learns to align RNA and MSI modalities
    in a shared embedding space using attention mechanisms.
    """
    # Transformer-based (100%)
    if trans_sim is not None and trans_sim.get('transformer_cosine', 0) != 0:
        trans_score = (
            0.70 * max(trans_sim['transformer_cosine'], 0) +
            0.30 * trans_sim['transformer_l2_similarity']
        )
    else:
        trans_score = 0
    
    return trans_score


# =============================================================================
# MAIN MATCHER
# =============================================================================

class HybridTransformerMatcher:
    """Hybrid matcher with Cross-Modal Transformer"""
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v10',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        
        subdirs = ['gene_visualizations', 'match_visualizations',
                   'training_curves', 'transformer_attention']
        for subdir in subdirs:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.cross_modal_transformer = None
    
    def load_all_data(self):
        """Load data"""
        print("Loading RNA-seq data...")
        print(f"  Pixel size: {RNA_PIXEL_SIZE} μm (hexagonal)")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print(f"\nLoading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def train_cross_modal_transformer(self, rna_samples, msi_samples, epochs=100, val_split=0.2):
        """Train Cross-Modal Transformer with validation tracking"""
        print("\nTraining Cross-Modal Transformer...")
        
        self.cross_modal_transformer = CrossModalTransformer(
            d_model=128,
            n_heads=8,
            n_layers=4,
            d_ff=512,
            dropout=0.1
        ).to(self.device)
        
        optimizer = torch.optim.AdamW(
            self.cross_modal_transformer.parameters(),
            lr=0.0005,
            weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        contrastive_loss = CrossModalContrastiveLoss(temperature=0.07)
        
        # Split into train/val
        common_samples = list(set(rna_samples.keys()) & set(msi_samples.keys()))
        n_val = max(1, int(len(common_samples) * val_split))
        val_samples = common_samples[-n_val:]
        train_samples = common_samples[:-n_val]
        
        print(f"  Train samples: {len(train_samples)}, Val samples: {len(val_samples)}")
        
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        best_model_state = None
        
        for epoch in range(epochs):
            # Training
            self.cross_modal_transformer.train()
            epoch_train_loss = 0
            n_train_batches = 0
            
            for sample_id in train_samples:
                rna_adata = self.rna_data[sample_id]
                msi_adata = self.msi_data[sample_id]
                
                # Get coordinates
                rna_coords = np.column_stack([
                    rna_adata.obs['x_um'].values,
                    rna_adata.obs['y_um'].values
                ])
                msi_coords = np.column_stack([
                    msi_adata.obs['x_um'].values,
                    msi_adata.obs['y_um'].values
                ])
                
                # Align RNA coordinates
                rna_coords_aligned = align_rna_to_msi(rna_coords, msi_coords)
                
                # Get features (subsample for efficiency)
                max_rna = min(500, rna_adata.shape[0])
                max_msi = min(1000, msi_adata.shape[0])
                
                rna_idx = np.random.choice(rna_adata.shape[0], max_rna, replace=False)
                msi_idx = np.random.choice(msi_adata.shape[0], max_msi, replace=False)
                
                rna_features = rna_adata.X[rna_idx, :50]
                if hasattr(rna_features, 'toarray'):
                    rna_features = rna_features.toarray()
                
                msi_features = msi_adata.X[msi_idx, :50]
                if hasattr(msi_features, 'toarray'):
                    msi_features = msi_features.toarray()
                
                # Scale features
                scaler = RobustScaler()
                rna_features = scaler.fit_transform(rna_features)
                msi_features = scaler.fit_transform(msi_features)
                
                # Convert to tensors
                rna_feat_t = torch.tensor(rna_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                rna_coord_t = torch.tensor(rna_coords_aligned[rna_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                msi_feat_t = torch.tensor(msi_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                msi_coord_t = torch.tensor(msi_coords[msi_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                
                # Forward pass - don't squeeze during training
                optimizer.zero_grad()
                outputs = self.cross_modal_transformer(
                    rna_feat_t, rna_coord_t, msi_feat_t, msi_coord_t,
                    return_squeezed=False
                )
                
                # Get embeddings - they should be [batch, d_model]
                rna_emb = outputs['rna_embedding']
                msi_emb = outputs['msi_embedding']
                
                # Contrastive loss on embeddings
                loss = contrastive_loss(rna_emb, msi_emb)
                
                # Add alignment loss
                sim_score = outputs['similarity_score']
                alignment_loss = 1 - sim_score.mean()
                loss = loss + 0.5 * alignment_loss
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.cross_modal_transformer.parameters(), 1.0)
                optimizer.step()
                
                epoch_train_loss += loss.item()
                n_train_batches += 1
            
            scheduler.step()
            train_losses.append(epoch_train_loss / max(n_train_batches, 1))
            
            # Validation
            self.cross_modal_transformer.eval()
            epoch_val_loss = 0
            n_val_batches = 0
            
            with torch.no_grad():
                for sample_id in val_samples:
                    rna_adata = self.rna_data[sample_id]
                    msi_adata = self.msi_data[sample_id]
                    
                    rna_coords = np.column_stack([
                        rna_adata.obs['x_um'].values,
                        rna_adata.obs['y_um'].values
                    ])
                    msi_coords = np.column_stack([
                        msi_adata.obs['x_um'].values,
                        msi_adata.obs['y_um'].values
                    ])
                    
                    rna_coords_aligned = align_rna_to_msi(rna_coords, msi_coords)
                    
                    max_rna = min(500, rna_adata.shape[0])
                    max_msi = min(1000, msi_adata.shape[0])
                    
                    rna_idx = np.random.choice(rna_adata.shape[0], max_rna, replace=False)
                    msi_idx = np.random.choice(msi_adata.shape[0], max_msi, replace=False)
                    
                    rna_features = rna_adata.X[rna_idx, :50]
                    if hasattr(rna_features, 'toarray'):
                        rna_features = rna_features.toarray()
                    
                    msi_features = msi_adata.X[msi_idx, :50]
                    if hasattr(msi_features, 'toarray'):
                        msi_features = msi_features.toarray()
                    
                    scaler = RobustScaler()
                    rna_features = scaler.fit_transform(rna_features)
                    msi_features = scaler.fit_transform(msi_features)
                    
                    rna_feat_t = torch.tensor(rna_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                    rna_coord_t = torch.tensor(rna_coords_aligned[rna_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                    msi_feat_t = torch.tensor(msi_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                    msi_coord_t = torch.tensor(msi_coords[msi_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                    
                    outputs = self.cross_modal_transformer(
                        rna_feat_t, rna_coord_t, msi_feat_t, msi_coord_t,
                        return_squeezed=False
                    )
                    
                    rna_emb = outputs['rna_embedding']
                    msi_emb = outputs['msi_embedding']
                    
                    loss = contrastive_loss(rna_emb, msi_emb)
                    sim_score = outputs['similarity_score']
                    alignment_loss = 1 - sim_score.mean()
                    loss = loss + 0.5 * alignment_loss
                    
                    epoch_val_loss += loss.item()
                    n_val_batches += 1
            
            val_losses.append(epoch_val_loss / max(n_val_batches, 1))
            
            # Save best model
            if val_losses[-1] < best_val_loss:
                best_val_loss = val_losses[-1]
                best_model_state = self.cross_modal_transformer.state_dict().copy()
            
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Train: {train_losses[-1]:.4f}, Val: {val_losses[-1]:.4f}")
        
        # Load best model
        if best_model_state is not None:
            self.cross_modal_transformer.load_state_dict(best_model_state)
            print(f"  Loaded best model with val loss: {best_val_loss:.4f}")
        
        # Plot train and validation curves
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss', color='blue')
        plt.plot(val_losses, label='Validation Loss', color='red', linestyle='--')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Cross-Modal Transformer Training')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Mark best validation point
        best_epoch = np.argmin(val_losses)
        plt.axvline(x=best_epoch, color='green', linestyle=':', alpha=0.7)
        plt.scatter([best_epoch], [val_losses[best_epoch]], color='green', s=100, zorder=5, label=f'Best Val (epoch {best_epoch})')
        
        plt.legend()
        plt.savefig(os.path.join(self.output_dir, 'training_curves', 'transformer_loss.png'), dpi=150)
        plt.close()
        
        print(f"  Final train loss: {train_losses[-1]:.4f}, Final val loss: {val_losses[-1]:.4f}")
    
    def extract_signature(
        self,
        coords: np.ndarray,
        values: np.ndarray,
        sample_id: str,
        feature_name: str,
        feature_type: str,
        msi_coords: Optional[np.ndarray] = None
    ) -> SpatialSignature:
        """Extract spatial signature"""
        
        aligned = None
        if feature_type == 'gene' and msi_coords is not None:
            aligned = align_rna_to_msi(coords, msi_coords)
        
        return SpatialSignature(
            sample_id=sample_id,
            feature_name=feature_name,
            feature_type=feature_type,
            transformer_embedding=None,
            coordinates=coords,
            raw_values=values,
            aligned_coordinates=aligned
        )
    
    def compute_transformer_embeddings(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature
    ) -> Tuple[np.ndarray, np.ndarray, dict]:
        """Compute Cross-Modal Transformer embeddings for a gene-mz pair"""
        
        if self.cross_modal_transformer is None:
            return None, None, {}
        
        self.cross_modal_transformer.eval()
        
        # Use aligned coordinates for gene
        gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
        mz_coords = mz_sig.coordinates
        
        # Prepare features
        gene_values = gene_sig.raw_values.reshape(-1, 1)
        mz_values = mz_sig.raw_values.reshape(-1, 1)
        
        scaler = RobustScaler()
        gene_scaled = scaler.fit_transform(gene_values)
        mz_scaled = scaler.fit_transform(mz_values)
        
        # Convert to tensors
        gene_feat_t = torch.tensor(gene_scaled, dtype=torch.float32, device=self.device).unsqueeze(0)
        gene_coord_t = torch.tensor(gene_coords, dtype=torch.float32, device=self.device).unsqueeze(0)
        mz_feat_t = torch.tensor(mz_scaled, dtype=torch.float32, device=self.device).unsqueeze(0)
        mz_coord_t = torch.tensor(mz_coords, dtype=torch.float32, device=self.device).unsqueeze(0)
        
        with torch.no_grad():
            outputs = self.cross_modal_transformer(gene_feat_t, gene_coord_t, mz_feat_t, mz_coord_t)
        
        attention_info = {
            'rna_to_msi': outputs['cross_attention_rna_to_msi'].cpu().numpy(),
            'msi_to_rna': outputs['cross_attention_msi_to_rna'].cpu().numpy(),
            'similarity': outputs['similarity_score'].cpu().numpy()
        }
        
        return (
            outputs['rna_embedding'].cpu().numpy(),
            outputs['msi_embedding'].cpu().numpy(),
            attention_info
        )
    
    def visualize_signature(self, sig: SpatialSignature, save_path: str):
        """Visualize gene/mz expression patterns"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # Row 0: Expression patterns
        # Original coordinates expression
        im1 = axes[0, 0].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.raw_values, cmap='viridis', s=10)
        axes[0, 0].set_title(f'{sig.feature_name} - Original', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0], label='Expression')
        
        # Aligned coordinates (for genes) or log-transformed
        if sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(sig.aligned_coordinates[:, 0], sig.aligned_coordinates[:, 1],
                                      c=sig.raw_values, cmap='viridis', s=10)
            axes[0, 1].set_title('Aligned (180° rotated)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='Expression')
        else:
            # Show log-transformed for better visualization
            log_vals = np.log1p(sig.raw_values)
            im2 = axes[0, 1].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                      c=log_vals, cmap='viridis', s=10)
            axes[0, 1].set_title('Log-transformed', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='log(1+x)')
        axes[0, 1].set_xlabel('X (μm)')
        axes[0, 1].set_ylabel('Y (μm)')
        
        # Expression percentiles spatial view
        coords = sig.aligned_coordinates if sig.aligned_coordinates is not None else sig.coordinates
        thresh_90 = np.percentile(sig.raw_values, 90)
        mask_90 = sig.raw_values >= thresh_90
        axes[0, 2].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=5, alpha=0.3)
        if mask_90.sum() > 0:
            im3 = axes[0, 2].scatter(coords[mask_90, 0], coords[mask_90, 1],
                                      c=sig.raw_values[mask_90], cmap='hot', s=15)
            plt.colorbar(im3, ax=axes[0, 2], label='Expression')
        axes[0, 2].set_title(f'Top 10% Expression ({mask_90.sum()} spots)', fontweight='bold')
        axes[0, 2].set_xlabel('X (μm)')
        axes[0, 2].set_ylabel('Y (μm)')
        
        # Row 1: More visualizations
        # Expression histogram
        axes[1, 0].hist(sig.raw_values, bins=50, color='steelblue', edgecolor='darkblue', alpha=0.7)
        axes[1, 0].set_title('Expression Distribution', fontweight='bold')
        axes[1, 0].set_xlabel('Expression Value')
        axes[1, 0].set_ylabel('Count')
        
        # Expression percentiles visualization
        percentiles = [25, 50, 75, 90]
        colors = ['lightblue', 'steelblue', 'orange', 'red']
        axes[1, 1].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=3, alpha=0.3)
        for pct, color in zip(percentiles, colors):
            thresh = np.percentile(sig.raw_values, pct)
            mask = sig.raw_values >= thresh
            axes[1, 1].scatter(coords[mask, 0], coords[mask, 1], 
                              c=color, s=5, alpha=0.5, label=f'≥{pct}th pct')
        axes[1, 1].set_title('Expression Percentiles', fontweight='bold')
        axes[1, 1].set_xlabel('X (μm)')
        axes[1, 1].set_ylabel('Y (μm)')
        axes[1, 1].legend(loc='upper right', fontsize=8)
        
        # Summary statistics
        stats_text = f"""
EXPRESSION STATISTICS
═══════════════════════════════

Feature: {sig.feature_name}
Sample:  {sig.sample_id}
Type:    {sig.feature_type.upper()}

Expression Values:
  Mean:   {sig.raw_values.mean():.4f}
  Std:    {sig.raw_values.std():.4f}
  Min:    {sig.raw_values.min():.4f}
  Max:    {sig.raw_values.max():.4f}
  Median: {np.median(sig.raw_values):.4f}

Spatial Metrics:
  Total Spots:   {len(sig.raw_values)}
  Non-zero:      {(sig.raw_values > 0).sum()}
  Sparsity:      {(sig.raw_values == 0).sum() / len(sig.raw_values) * 100:.1f}%
        """
        axes[1, 2].text(0.05, 0.95, stats_text, transform=axes[1, 2].transAxes,
                       fontsize=10, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))
        axes[1, 2].axis('off')
        
        plt.suptitle(f'EXPRESSION PATTERN: {sig.feature_name} ({sig.sample_id})',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_match(self, gene_sig, mz_sig, trans_sim, save_path):
        """Visualize match with transformer attention"""
        combined = compute_combined_score(trans_sim)
        
        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        
        # Row 0: Gene
        im1 = axes[0, 0].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        if gene_sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                                      c=gene_sig.raw_values, cmap='viridis', s=15)
            axes[0, 1].set_title('Gene (180° Aligned)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1])
        
        # Gene top expression
        gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
        gene_thresh = np.percentile(gene_sig.raw_values, 90)
        gene_mask = gene_sig.raw_values >= gene_thresh
        axes[0, 2].scatter(gene_coords[:, 0], gene_coords[:, 1], c='lightgray', s=5, alpha=0.3)
        if gene_mask.sum() > 0:
            im3 = axes[0, 2].scatter(gene_coords[gene_mask, 0], gene_coords[gene_mask, 1],
                                      c=gene_sig.raw_values[gene_mask], cmap='hot', s=15)
            plt.colorbar(im3, ax=axes[0, 2])
        axes[0, 2].set_title('Gene Top 10% Expression', fontweight='bold')
        
        # Row 1: m/z
        im4 = axes[1, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.raw_values, cmap='viridis', s=5)
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 0])
        
        # m/z top expression
        mz_thresh = np.percentile(mz_sig.raw_values, 90)
        mz_mask = mz_sig.raw_values >= mz_thresh
        axes[1, 1].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1], c='lightgray', s=3, alpha=0.3)
        if mz_mask.sum() > 0:
            im5 = axes[1, 1].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                                      c=mz_sig.raw_values[mz_mask], cmap='hot', s=8)
            plt.colorbar(im5, ax=axes[1, 1])
        axes[1, 1].set_title('m/z Top 10% Expression', fontweight='bold')
        
        # Expression histograms comparison
        axes[1, 2].hist(gene_sig.raw_values, bins=30, alpha=0.5, label='Gene', color='blue')
        axes[1, 2].hist(mz_sig.raw_values, bins=30, alpha=0.5, label='m/z', color='red')
        axes[1, 2].set_title('Expression Distributions', fontweight='bold')
        axes[1, 2].legend()
        
        # Row 2: Overlays and metrics
        if gene_sig.aligned_coordinates is not None:
            axes[2, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                              c='blue', s=3, alpha=0.3, label='m/z')
            axes[2, 0].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                              c='red', s=10, alpha=0.5, label='Gene')
            axes[2, 0].set_title('Overlay (Gene aligned)', fontweight='bold')
            axes[2, 0].legend()
        
        # Top expression overlay
        if gene_sig.aligned_coordinates is not None:
            axes[2, 1].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                              c='blue', s=8, alpha=0.5, label='m/z top 10%')
            axes[2, 1].scatter(gene_coords[gene_mask, 0], gene_coords[gene_mask, 1],
                              c='red', s=15, alpha=0.7, label='Gene top 10%')
            axes[2, 1].set_title('Top 10% Overlay', fontweight='bold')
            axes[2, 1].legend()
        
        # Metrics with transformer scores
        trans_cosine = trans_sim.get('transformer_cosine', 0) if trans_sim else 0
        trans_l2 = trans_sim.get('transformer_l2_similarity', 0) if trans_sim else 0
        
        metrics_text = f"""
COMBINED SCORE: {combined:.4f}
═══════════════════════════════════

TRANSFORMER-BASED (100%):
  Transformer cosine:   {trans_cosine:.4f}
  Transformer L2 sim:   {trans_l2:.4f}
        """
        axes[2, 2].text(0.05, 0.95, metrics_text, transform=axes[2, 2].transAxes,
                       fontsize=9, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[2, 2].axis('off')
        
        plt.suptitle(f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name} | Score: {combined:.3f}',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def find_matches(self, gene_sig, mz_sigs, top_k=20):
        """Find matches using transformer similarity"""
        matches = []
        
        for mz_name, mz_sig in mz_sigs.items():
            # Compute transformer similarity
            if self.cross_modal_transformer is not None:
                gene_emb, mz_emb, attn_info = self.compute_transformer_embeddings(gene_sig, mz_sig)
                
                if gene_emb is not None:
                    gene_sig.transformer_embedding = gene_emb
                    mz_sig.transformer_embedding = mz_emb
                
                trans_sim = compute_transformer_similarity(gene_sig, mz_sig)
            else:
                trans_sim = {'transformer_cosine': 0, 'transformer_l2_similarity': 0}
            
            combined = compute_combined_score(trans_sim)
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                **trans_sim,
                'combined_score': combined
            })
        
        return pd.DataFrame(matches).sort_values('combined_score', ascending=False).head(top_k)
    
    def run_analysis(self, epochs_transformer=100, top_k=20):
        """Run analysis with Cross-Modal Transformer"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V10")
        print(f"RNA: {RNA_PIXEL_SIZE}μm (hexagonal) | MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print("Transformer-Only Matching")
        print("="*70)
        
        # Gene availability
        gene_avail = {}
        for gene in AAD_TARGET_GENES:
            gene_avail[gene] = {s: gene in self.rna_data[s].var_names 
                               for s in RNA_SAMPLE_IDS if s in self.rna_data}
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            n = sum(gene_avail[gene].values())
            print(f"  {gene}: {n}/{len(RNA_SAMPLE_IDS)}")
        
        # Prepare data for transformer training
        print("\n" + "-"*70)
        print("PHASE 1: Preparing Data for Transformer")
        print("-"*70)
        
        rna_train = {}
        for sid in list(self.rna_data.keys())[:4]:
            adata = self.rna_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            rna_train[sid] = {'coords': coords, 'features': features}
            print(f"  RNA {sid}: {coords.shape[0]} spots")
        
        msi_train = {}
        for sid in list(self.msi_data.keys())[:4]:
            adata = self.msi_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            msi_train[sid] = {'coords': coords, 'features': features}
            print(f"  MSI {sid}: {coords.shape[0]} spots")
        
        # Train Cross-Modal Transformer
        print("\n" + "-"*70)
        print("PHASE 2: Training Cross-Modal Transformer")
        print("-"*70)
        self.train_cross_modal_transformer(rna_train, msi_train, epochs_transformer)
        
        # Match
        print("\n" + "-"*70)
        print("PHASE 3: Matching (All Samples)")
        print("-"*70)
        
        all_results = []
        all_top5_results = []  # Store top 5 matches per gene-sample for detailed CSV     
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_avail[gene].items() if a]
            if not available:
                continue
            
            for rna_sample in available:
                print(f"\n  {rna_sample}")
                
                adata = self.rna_data[rna_sample]
                rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                gene_idx = list(adata.var_names).index(gene)
                gene_expr = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, 'toarray') \
                           else adata.X[:, gene_idx].flatten()
                
                msi_sample = rna_sample
                if msi_sample not in self.msi_data:
                    print(f"    MSI {msi_sample} not found")
                    continue
                
                msi_adata = self.msi_data[msi_sample]
                msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
                
                gene_sig = self.extract_signature(
                    rna_coords, gene_expr, rna_sample, gene, 'gene', msi_coords
                )
                
                # Gene visualization - focuses on expression patterns
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'gene_visualizations', f'{gene}_{rna_sample}.png')
                )

                print(f"    Matching vs MSI...")
                msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
                mz_names = list(msi_adata.var_names)
                
                mz_sigs = {}
                for i, mz_name in enumerate(mz_names):
                    mz_sigs[mz_name] = self.extract_signature(
                        msi_coords, msi_data[:, i], msi_sample, mz_name, 'mz'
                    )
                    if (i + 1) % 100 == 0:
                        print(f"      {i+1}/{len(mz_names)}")
                
                matches = self.find_matches(gene_sig, mz_sigs, top_k)
                all_results.append(matches)


                # Store top 5 for the detailed CSV
                top5_matches = matches.head(5).copy()
                top5_matches['gene'] = gene
                top5_matches['rna_sample'] = rna_sample
                top5_matches['rank'] = range(1, len(top5_matches) + 1)
                all_top5_results.append(top5_matches)
                
                if len(matches) > 0:
                    print(f"\n    Top 5:")
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        trans_score = m.get('transformer_cosine', 0)
                        print(f"      {idx+1}. m/z {m['mz_feature']}: {m['combined_score']:.3f} (trans: {trans_score:.3f})")
                    
                    # Visualize top match        
                    top = matches.iloc[0]
                    top_mz = mz_sigs[top['mz_feature']]
                    trans_sim = compute_transformer_similarity(gene_sig, top_mz)
                    
                    self.visualize_match(
                        gene_sig, top_mz, trans_sim,
                        os.path.join(self.output_dir, 'match_visualizations', f'{gene}_{rna_sample}_top.png')
                    )
        
        # Save results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            results.to_csv(os.path.join(self.output_dir, 'gene_to_mz_matches_v10.csv'), index=False)
             
            # Save top 5 matches per gene-sample with all scores
            if all_top5_results:
                top5_df = pd.concat(all_top5_results, ignore_index=True)
                
                # Reorder columns for better readability
                priority_cols = ['gene', 'rna_sample', 'rank', 'mz_feature', 'msi_sample', 'combined_score']
                other_cols = [c for c in top5_df.columns if c not in priority_cols]
                top5_df = top5_df[priority_cols + other_cols]
                
                # Sort by gene, sample, then rank
                top5_df = top5_df.sort_values(['gene', 'rna_sample', 'rank'])
                
                top5_df.to_csv(os.path.join(self.output_dir, 'gene_to_mz_top5_matches_all_scores.csv'), index=False)
                print(f"\nSaved top 5 matches per gene-sample to: gene_to_mz_top5_matches_all_scores.csv")
                print(f"  Total entries: {len(top5_df)}")
                print(f"  Genes: {top5_df['gene'].nunique()}")
                print(f"  Samples: {top5_df['rna_sample'].nunique()}")
            
            print("\n" + "="*70)
            print("TOP MATCHES")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gr = results[results['gene'] == gene]
                if len(gr) > 0:
                    t = gr.iloc[0]
                    trans_score = t.get('transformer_cosine', 0)
                    print(f"\n{gene}: m/z {t['mz_feature']} ({t['rna_sample']}) = {t['combined_score']:.3f} (trans: {trans_score:.3f})")
            
            return results
        
        return None
    


def main():
    print("="*70)
    print("V10: Transformer-Only Matching")
    print(f"RNA: {RNA_PIXEL_SIZE}μm | MSI: {MSI_PIXEL_SIZE}μm")
    print("="*70)
    
    matcher = HybridTransformerMatcher(output_dir='./gene_to_mz_results_v10')
    matcher.load_all_data()
    results = matcher.run_analysis(epochs_transformer=100, top_k=20)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    
    return matcher, results


if __name__ == "__main__":
    matcher, results = main()

V10: Transformer-Only Matching
RNA: 55μm | MSI: 60μm
Loading RNA-seq data...
  Pixel size: 55 μm (hexagonal)
  YC_1: (2112, 800)
  YC_2: (2775, 800)
  YC_3: (2808, 800)
  YC_4: (2725, 800)
  YAD_1: (2915, 800)
  YAD_2: (2960, 800)
  YAD_3: (2880, 800)
  YAD_4: (2939, 800)
  AC_1: (3065, 800)
  AC_2: (3054, 800)
  AC_3: (2892, 800)
  AC_4: (3002, 800)
  AAD_1: (2700, 800)
  AAD_2: (2171, 800)
  AAD_3: (1584, 800)
  AAD_4: (2438, 800)

Loading MSI data...
  Pixel size: 60 μm (Cartesian)
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

GENE-TO-M/Z MATCHING V10
RNA: 55μm (hexagonal) | MSI: 60μm (Cartesian)
Transformer-Only Matching

Gene availability:
  Thy1: 16/16
  App: 16/16
  Apoe: 16/16
  Mapt: 16/16

--------

KeyboardInterrupt: 

In [ ]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V10
=================================================

KEY PARAMETERS:
- RNA (Visium): ~55 μm pixel size, hexagonal grid, 6 neighbors
- MSI: 60 μm pixel size, Cartesian grid, 8 neighbors
- RNA needs 180° rotation to align with MSI

APPROACH:
This version uses Cross-Modal Transformers exclusively for similarity computation.
The transformer learns to align RNA and MSI modalities in a shared embedding space
using attention mechanisms and learned positional encodings.

Changes from V9:
- Removed coordinate-based similarity computation
- Removed descriptor-based similarity computation  
- Removed GNN models entirely
- Combined score now uses only transformer-based metrics (100%)

Author: V10 - Transformer-Only Similarity
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
from typing import Dict, Tuple, Optional
import pandas as pd
import os
import warnings
from dataclasses import dataclass
import pickle
import math

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

RNA_PIXEL_SIZE = 55  # μm (Visium)
MSI_PIXEL_SIZE = 60  # μm

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = ['Thy1', 'App', 'Apoe', 'Mapt']


# =============================================================================
# COORDINATE TRANSFORMATION
# =============================================================================

def rotate_180(coords: np.ndarray) -> np.ndarray:
    """Rotate coordinates 180° around centroid"""
    center = coords.mean(axis=0)
    return 2 * center - coords


def align_rna_to_msi(rna_coords: np.ndarray, msi_coords: np.ndarray) -> np.ndarray:
    """Align RNA coordinates to MSI coordinate system with 180° rotation and scaling"""
    rotated = rotate_180(rna_coords)
    
    rna_min, rna_max = rotated.min(axis=0), rotated.max(axis=0)
    msi_min, msi_max = msi_coords.min(axis=0), msi_coords.max(axis=0)
    
    rna_range = rna_max - rna_min
    msi_range = msi_max - msi_min
    scale = msi_range / (rna_range + 1e-8)
    
    return (rotated - rna_min) * scale + msi_min


# =============================================================================
# LEARNED POSITIONAL ENCODING
# =============================================================================

class LearnedPositionalEncoding2D(nn.Module):
    """
    Learned 2D positional encoding for spatial coordinates.
    
    Combines learnable embeddings with sinusoidal features for both
    absolute position and relative spatial relationships.
    """
    
    def __init__(self, d_model: int, max_spatial_size: int = 1000, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.max_spatial_size = max_spatial_size
        
        # Learnable position embeddings (grid-based)
        self.x_embedding = nn.Embedding(max_spatial_size, d_model // 2)
        self.y_embedding = nn.Embedding(max_spatial_size, d_model // 2)
        
        # Continuous coordinate projection (for non-grid positions)
        self.coord_projection = nn.Sequential(
            nn.Linear(2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
        # Learnable scale parameters for combining embeddings
        self.alpha = nn.Parameter(torch.ones(1) * 0.5)
        self.beta = nn.Parameter(torch.ones(1) * 0.5)
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        
    def _get_sinusoidal_encoding(self, coords: torch.Tensor) -> torch.Tensor:
        """Generate sinusoidal positional encoding from continuous coordinates"""
        batch_size, n_points, _ = coords.shape
        device = coords.device
        
        # Scale coordinates to [0, 1]
        coords_norm = coords.clone()
        for b in range(batch_size):
            c_min = coords[b].min(dim=0)[0]
            c_max = coords[b].max(dim=0)[0]
            coords_norm[b] = (coords[b] - c_min) / (c_max - c_min + 1e-8)
        
        # Generate frequency bands
        d_half = self.d_model // 4
        freq_bands = torch.exp(
            torch.arange(0, d_half, dtype=torch.float32, device=device) *
            (-math.log(10000.0) / d_half)
        )
        
        # Apply sinusoidal encoding to x and y separately
        x_enc = coords_norm[..., 0:1] * freq_bands  # [B, N, d_half]
        y_enc = coords_norm[..., 1:2] * freq_bands  # [B, N, d_half]
        
        pe = torch.cat([
            torch.sin(x_enc), torch.cos(x_enc),
            torch.sin(y_enc), torch.cos(y_enc)
        ], dim=-1)
        
        return pe
    
    def forward(self, x: torch.Tensor, coords: torch.Tensor) -> torch.Tensor:
        """
        Add positional encoding to input features.
        
        Args:
            x: Input features [batch_size, n_points, d_model]
            coords: Spatial coordinates [batch_size, n_points, 2]
        
        Returns:
            Position-encoded features [batch_size, n_points, d_model]
        """
        # Handle unbatched input
        if x.dim() == 2:
            x = x.unsqueeze(0)
            coords = coords.unsqueeze(0)
            squeeze_output = True
        else:
            squeeze_output = False
        
        batch_size, n_points, _ = x.shape
        device = x.device
        
        # Normalize coordinates to grid indices
        coords_norm = coords.clone()
        for b in range(batch_size):
            c_min = coords[b].min(dim=0)[0]
            c_max = coords[b].max(dim=0)[0]
            coords_norm[b] = (coords[b] - c_min) / (c_max - c_min + 1e-8)
        
        # Get grid indices (clamped to valid range)
        x_idx = (coords_norm[..., 0] * (self.max_spatial_size - 1)).long().clamp(0, self.max_spatial_size - 1)
        y_idx = (coords_norm[..., 1] * (self.max_spatial_size - 1)).long().clamp(0, self.max_spatial_size - 1)
        
        # Learnable grid embeddings
        x_emb = self.x_embedding(x_idx)  # [B, N, d_model//2]
        y_emb = self.y_embedding(y_idx)  # [B, N, d_model//2]
        grid_pe = torch.cat([x_emb, y_emb], dim=-1)  # [B, N, d_model]
        
        # Continuous coordinate projection
        coord_pe = self.coord_projection(coords)  # [B, N, d_model]
        
        # Sinusoidal encoding
        sin_pe = self._get_sinusoidal_encoding(coords)  # [B, N, d_model]
        
        # Combine all positional encodings with learnable weights
        alpha = torch.sigmoid(self.alpha)
        beta = torch.sigmoid(self.beta)
        
        combined_pe = alpha * grid_pe + beta * coord_pe + (1 - alpha - beta).clamp(min=0) * sin_pe
        
        # Add to input and normalize
        output = self.norm(x + self.dropout(combined_pe))
        
        if squeeze_output:
            output = output.squeeze(0)
        
        return output


# =============================================================================
# CROSS-MODAL TRANSFORMER
# =============================================================================

class CrossModalAttention(nn.Module):
    """
    Cross-modal attention layer for aligning RNA and MSI modalities.
    
    Uses multi-head attention where queries come from one modality
    and keys/values come from the other modality.
    """
    
    def __init__(self, d_model: int, n_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Cross-modal attention.
        
        Args:
            query: Query tensor [batch, n_query, d_model]
            key: Key tensor [batch, n_key, d_model]
            value: Value tensor [batch, n_key, d_model]
            mask: Optional attention mask
        
        Returns:
            output: Attended features [batch, n_query, d_model]
            attention: Attention weights [batch, n_heads, n_query, n_key]
        """
        batch_size = query.size(0)
        
        # Linear projections
        Q = self.w_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)
        
        # Apply attention to values
        context = torch.matmul(attention, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        output = self.w_o(context)
        
        return output, attention


class CrossModalTransformerLayer(nn.Module):
    """
    Single layer of the Cross-Modal Transformer.
    
    Contains:
    - Self-attention within each modality
    - Cross-attention between modalities
    - Feed-forward network
    """
    
    def __init__(self, d_model: int, n_heads: int = 8, d_ff: int = 512, dropout: float = 0.1):
        super().__init__()
        
        # Self-attention
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.self_norm = nn.LayerNorm(d_model)
        
        # Cross-attention
        self.cross_attn = CrossModalAttention(d_model, n_heads, dropout)
        self.cross_norm = nn.LayerNorm(d_model)
        
        # Feed-forward
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.ff_norm = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(
        self,
        x: torch.Tensor,
        context: torch.Tensor,
        x_coords: Optional[torch.Tensor] = None,
        context_coords: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through transformer layer.
        
        Args:
            x: Input features from source modality [batch, n_source, d_model]
            context: Features from target modality [batch, n_target, d_model]
            x_coords: Optional spatial coordinates for source
            context_coords: Optional spatial coordinates for target
        
        Returns:
            output: Transformed features [batch, n_source, d_model]
            cross_attention: Cross-modal attention weights
        """
        # Self-attention
        self_out, _ = self.self_attn(x, x, x)
        x = self.self_norm(x + self.dropout(self_out))
        
        # Cross-attention
        cross_out, cross_attn = self.cross_attn(x, context, context)
        x = self.cross_norm(x + self.dropout(cross_out))
        
        # Feed-forward
        ff_out = self.ff(x)
        x = self.ff_norm(x + ff_out)
        
        return x, cross_attn


class CrossModalTransformer(nn.Module):
    """
    Cross-Modal Transformer for aligning RNA and MSI spatial patterns.
    
    Uses learned positional encodings and bidirectional cross-attention
    to learn a shared embedding space between modalities.
    """
    
    def __init__(
        self,
        d_model: int = 128,
        n_heads: int = 8,
        n_layers: int = 4,
        d_ff: int = 512,
        dropout: float = 0.1,
        max_spatial_size: int = 1000
    ):
        super().__init__()
        
        self.d_model = d_model
        
        # Modality-specific input projections (will be created dynamically)
        self.rna_projections = nn.ModuleDict()
        self.msi_projections = nn.ModuleDict()
        
        # Learned positional encodings for each modality
        self.rna_pos_enc = LearnedPositionalEncoding2D(d_model, max_spatial_size, dropout)
        self.msi_pos_enc = LearnedPositionalEncoding2D(d_model, max_spatial_size, dropout)
        
        # Modality embeddings (learnable tokens to distinguish RNA vs MSI)
        self.rna_modality_emb = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.msi_modality_emb = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        
        # Transformer layers (RNA → MSI direction)
        self.rna_to_msi_layers = nn.ModuleList([
            CrossModalTransformerLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # Transformer layers (MSI → RNA direction)
        self.msi_to_rna_layers = nn.ModuleList([
            CrossModalTransformerLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        # Global pooling attention
        self.pool_attention = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, 1)
        )
        
        # Output projection to shared embedding space
        self.output_proj = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model)
        )
        
    def _get_projection(self, dim: int, modality: str) -> nn.Module:
        """Get or create input projection for given dimension"""
        key = str(dim)
        projections = self.rna_projections if modality == 'rna' else self.msi_projections
        
        if key not in projections:
            device = next(self.parameters()).device
            projections[key] = nn.Sequential(
                nn.Linear(dim, self.d_model),
                nn.LayerNorm(self.d_model),
                nn.GELU()
            ).to(device)
        
        return projections[key]
    
    def _global_pool(self, x: torch.Tensor) -> torch.Tensor:
        """Attention-based global pooling"""
        attn_weights = F.softmax(self.pool_attention(x).squeeze(-1), dim=-1)
        return (attn_weights.unsqueeze(-1) * x).sum(dim=-2)
    
    def forward(
        self,
        rna_features: torch.Tensor,
        rna_coords: torch.Tensor,
        msi_features: torch.Tensor,
        msi_coords: torch.Tensor,
        return_squeezed: bool = True
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass through cross-modal transformer.
        
        Args:
            rna_features: RNA features [batch, n_rna, d_rna] or [n_rna, d_rna]
            rna_coords: RNA spatial coordinates [batch, n_rna, 2] or [n_rna, 2]
            msi_features: MSI features [batch, n_msi, d_msi] or [n_msi, d_msi]
            msi_coords: MSI spatial coordinates [batch, n_msi, 2] or [n_msi, 2]
            return_squeezed: If True and input was unbatched, squeeze output
        
        Returns:
            Dictionary containing:
            - rna_embedding: Global RNA embedding
            - msi_embedding: Global MSI embedding
            - rna_node_embeddings: Per-node RNA embeddings
            - msi_node_embeddings: Per-node MSI embeddings
            - cross_attention_rna_to_msi: Attention weights RNA→MSI
            - cross_attention_msi_to_rna: Attention weights MSI→RNA
            - similarity_score: Cross-modal similarity score
        """
        # Handle unbatched input
        input_was_unbatched = rna_features.dim() == 2
        if input_was_unbatched:
            rna_features = rna_features.unsqueeze(0)
            rna_coords = rna_coords.unsqueeze(0)
            msi_features = msi_features.unsqueeze(0)
            msi_coords = msi_coords.unsqueeze(0)
        
        batch_size = rna_features.size(0)
        
        # Project inputs to common dimension
        rna_proj = self._get_projection(rna_features.size(-1), 'rna')
        msi_proj = self._get_projection(msi_features.size(-1), 'msi')
        
        rna_h = rna_proj(rna_features)
        msi_h = msi_proj(msi_features)
        
        # Add positional encodings
        rna_h = self.rna_pos_enc(rna_h, rna_coords)
        msi_h = self.msi_pos_enc(msi_h, msi_coords)
        
        # Add modality embeddings
        rna_h = rna_h + self.rna_modality_emb.expand(batch_size, rna_h.size(1), -1)
        msi_h = msi_h + self.msi_modality_emb.expand(batch_size, msi_h.size(1), -1)
        
        # Store attention weights
        rna_to_msi_attns = []
        msi_to_rna_attns = []
        
        # Process through transformer layers (bidirectional)
        rna_out = rna_h
        msi_out = msi_h
        
        for rna_layer, msi_layer in zip(self.rna_to_msi_layers, self.msi_to_rna_layers):
            # RNA attends to MSI
            rna_out, rna_attn = rna_layer(rna_out, msi_out, rna_coords, msi_coords)
            rna_to_msi_attns.append(rna_attn)
            
            # MSI attends to RNA
            msi_out, msi_attn = msi_layer(msi_out, rna_out, msi_coords, rna_coords)
            msi_to_rna_attns.append(msi_attn)
        
        # Global pooling
        rna_global = self._global_pool(rna_out)
        msi_global = self._global_pool(msi_out)
        
        # Combined embedding
        combined = torch.cat([rna_global, msi_global], dim=-1)
        shared_embedding = self.output_proj(combined)
        
        # Compute similarity score
        rna_norm = F.normalize(rna_global, p=2, dim=-1)
        msi_norm = F.normalize(msi_global, p=2, dim=-1)
        similarity = (rna_norm * msi_norm).sum(dim=-1)
        
        outputs = {
            'rna_embedding': rna_global,
            'msi_embedding': msi_global,
            'shared_embedding': shared_embedding,
            'rna_node_embeddings': rna_out,
            'msi_node_embeddings': msi_out,
            'cross_attention_rna_to_msi': torch.stack(rna_to_msi_attns),
            'cross_attention_msi_to_rna': torch.stack(msi_to_rna_attns),
            'similarity_score': similarity
        }
        
        # Only squeeze if input was unbatched AND caller wants squeezed output
        if input_was_unbatched and return_squeezed:
            for key in ['rna_node_embeddings', 'msi_node_embeddings', 
                        'cross_attention_rna_to_msi', 'cross_attention_msi_to_rna']:
                if key in outputs and isinstance(outputs[key], torch.Tensor):
                    if outputs[key].dim() > 0 and outputs[key].size(0) == 1:
                        outputs[key] = outputs[key].squeeze(0)
        
        return outputs


class CrossModalContrastiveLoss(nn.Module):
    """
    Contrastive loss for training the Cross-Modal Transformer.
    
    Encourages matching RNA-MSI pairs to have similar embeddings
    while pushing non-matching pairs apart.
    """
    
    def __init__(self, temperature: float = 0.07, margin: float = 0.5):
        super().__init__()
        self.temperature = temperature
        self.margin = margin
        
    def forward(
        self,
        rna_embeddings: torch.Tensor,
        msi_embeddings: torch.Tensor,
        labels: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Compute contrastive loss.
        
        Args:
            rna_embeddings: RNA embeddings [batch, d_model]
            msi_embeddings: MSI embeddings [batch, d_model]
            labels: Optional matching labels (1 for match, 0 for non-match)
        
        Returns:
            Contrastive loss value
        """
        # Ensure 2D tensors
        if rna_embeddings.dim() == 1:
            rna_embeddings = rna_embeddings.unsqueeze(0)
        if msi_embeddings.dim() == 1:
            msi_embeddings = msi_embeddings.unsqueeze(0)
        
        batch_size = rna_embeddings.size(0)
        
        # Normalize embeddings
        rna_norm = F.normalize(rna_embeddings, p=2, dim=-1)
        msi_norm = F.normalize(msi_embeddings, p=2, dim=-1)
        
        # For single sample, use cosine similarity loss directly
        if batch_size == 1:
            # Maximize similarity for matching pairs
            similarity = (rna_norm * msi_norm).sum(dim=-1)
            loss = 1 - similarity.mean()
            return loss
        
        # For batch, compute similarity matrix
        similarity = torch.matmul(rna_norm, msi_norm.T) / self.temperature
        
        if labels is None:
            # Assume diagonal elements are positive pairs
            labels = torch.eye(batch_size, device=rna_embeddings.device)
        
        # InfoNCE loss
        log_softmax = F.log_softmax(similarity, dim=-1)
        loss = -(labels * log_softmax).sum(dim=-1).mean()
        
        return loss


# =============================================================================
# SPATIAL SIGNATURE (EXTENDED)
# =============================================================================

@dataclass
class SpatialSignature:
    """Spatial signature with transformer embeddings"""
    sample_id: str
    feature_name: str
    feature_type: str
    
    # Cross-Modal Transformer embeddings
    transformer_embedding: Optional[np.ndarray] = None
    
    # Raw data
    coordinates: np.ndarray = None
    raw_values: np.ndarray = None
    
    # Aligned coordinates
    aligned_coordinates: Optional[np.ndarray] = None


# =============================================================================
# SIMILARITY COMPUTATION
# =============================================================================

def compute_transformer_similarity(gene_sig: SpatialSignature, mz_sig: SpatialSignature) -> dict:
    """Compute similarity using Cross-Modal Transformer embeddings"""
    
    if gene_sig.transformer_embedding is None or mz_sig.transformer_embedding is None:
        return {
            'transformer_cosine': 0,
            'transformer_l2_distance': 1.0
        }
    
    g_emb = gene_sig.transformer_embedding.flatten()
    m_emb = mz_sig.transformer_embedding.flatten()
    
    # Cosine similarity
    cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
    
    # L2 distance (normalized)
    l2_dist = np.linalg.norm(g_emb - m_emb)
    l2_sim = 1 / (1 + l2_dist)
    
    return {
        'transformer_cosine': cosine,
        'transformer_l2_similarity': l2_sim
    }


def compute_combined_score(trans_sim: dict) -> float:
    """
    Combined score using transformer-based metrics only.
    
    The Cross-Modal Transformer learns to align RNA and MSI modalities
    in a shared embedding space using attention mechanisms.
    """
    # Transformer-based (100%)
    if trans_sim is not None and trans_sim.get('transformer_cosine', 0) != 0:
        trans_score = (
            0.70 * max(trans_sim['transformer_cosine'], 0) +
            0.30 * trans_sim['transformer_l2_similarity']
        )
    else:
        trans_score = 0
    
    return trans_score


# =============================================================================
# MAIN MATCHER
# =============================================================================

class HybridTransformerMatcher:
    """Hybrid matcher with Cross-Modal Transformer"""
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v10',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        
        subdirs = ['gene_visualizations', 'match_visualizations',
                   'training_curves', 'transformer_attention']
        for subdir in subdirs:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.cross_modal_transformer = None
    
    def load_all_data(self):
        """Load data"""
        print("Loading RNA-seq data...")
        print(f"  Pixel size: {RNA_PIXEL_SIZE} μm (hexagonal)")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print(f"\nLoading MSI data...")
        print(f"  Pixel size: {MSI_PIXEL_SIZE} μm (Cartesian)")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def train_cross_modal_transformer(self, rna_samples, msi_samples, epochs=100, val_split=0.2):
        """Train Cross-Modal Transformer with validation tracking"""
        print("\nTraining Cross-Modal Transformer...")
        
        self.cross_modal_transformer = CrossModalTransformer(
            d_model=128,
            n_heads=8,
            n_layers=4,
            d_ff=512,
            dropout=0.1
        ).to(self.device)
        
        optimizer = torch.optim.AdamW(
            self.cross_modal_transformer.parameters(),
            lr=0.0005,
            weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        contrastive_loss = CrossModalContrastiveLoss(temperature=0.07)
        
        # Split into train/val
        common_samples = list(set(rna_samples.keys()) & set(msi_samples.keys()))
        n_val = max(1, int(len(common_samples) * val_split))
        val_samples = common_samples[-n_val:]
        train_samples = common_samples[:-n_val]
        
        print(f"  Train samples: {len(train_samples)}, Val samples: {len(val_samples)}")
        
        train_losses = []
        val_losses = []
        best_val_loss = float('inf')
        best_model_state = None
        
        for epoch in range(epochs):
            # Training
            self.cross_modal_transformer.train()
            epoch_train_loss = 0
            n_train_batches = 0
            
            for sample_id in train_samples:
                rna_adata = self.rna_data[sample_id]
                msi_adata = self.msi_data[sample_id]
                
                # Get coordinates
                rna_coords = np.column_stack([
                    rna_adata.obs['x_um'].values,
                    rna_adata.obs['y_um'].values
                ])
                msi_coords = np.column_stack([
                    msi_adata.obs['x_um'].values,
                    msi_adata.obs['y_um'].values
                ])
                
                # Align RNA coordinates
                rna_coords_aligned = align_rna_to_msi(rna_coords, msi_coords)
                
                # Get features (subsample for efficiency)
                max_rna = min(500, rna_adata.shape[0])
                max_msi = min(1000, msi_adata.shape[0])
                
                rna_idx = np.random.choice(rna_adata.shape[0], max_rna, replace=False)
                msi_idx = np.random.choice(msi_adata.shape[0], max_msi, replace=False)
                
                rna_features = rna_adata.X[rna_idx, :50]
                if hasattr(rna_features, 'toarray'):
                    rna_features = rna_features.toarray()
                
                msi_features = msi_adata.X[msi_idx, :50]
                if hasattr(msi_features, 'toarray'):
                    msi_features = msi_features.toarray()
                
                # Scale features
                scaler = RobustScaler()
                rna_features = scaler.fit_transform(rna_features)
                msi_features = scaler.fit_transform(msi_features)
                
                # Convert to tensors
                rna_feat_t = torch.tensor(rna_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                rna_coord_t = torch.tensor(rna_coords_aligned[rna_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                msi_feat_t = torch.tensor(msi_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                msi_coord_t = torch.tensor(msi_coords[msi_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                
                # Forward pass - don't squeeze during training
                optimizer.zero_grad()
                outputs = self.cross_modal_transformer(
                    rna_feat_t, rna_coord_t, msi_feat_t, msi_coord_t,
                    return_squeezed=False
                )
                
                # Get embeddings - they should be [batch, d_model]
                rna_emb = outputs['rna_embedding']
                msi_emb = outputs['msi_embedding']
                
                # Contrastive loss on embeddings
                loss = contrastive_loss(rna_emb, msi_emb)
                
                # Add alignment loss
                sim_score = outputs['similarity_score']
                alignment_loss = 1 - sim_score.mean()
                loss = loss + 0.5 * alignment_loss
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.cross_modal_transformer.parameters(), 1.0)
                optimizer.step()
                
                epoch_train_loss += loss.item()
                n_train_batches += 1
            
            scheduler.step()
            train_losses.append(epoch_train_loss / max(n_train_batches, 1))
            
            # Validation
            self.cross_modal_transformer.eval()
            epoch_val_loss = 0
            n_val_batches = 0
            
            with torch.no_grad():
                for sample_id in val_samples:
                    rna_adata = self.rna_data[sample_id]
                    msi_adata = self.msi_data[sample_id]
                    
                    rna_coords = np.column_stack([
                        rna_adata.obs['x_um'].values,
                        rna_adata.obs['y_um'].values
                    ])
                    msi_coords = np.column_stack([
                        msi_adata.obs['x_um'].values,
                        msi_adata.obs['y_um'].values
                    ])
                    
                    rna_coords_aligned = align_rna_to_msi(rna_coords, msi_coords)
                    
                    max_rna = min(500, rna_adata.shape[0])
                    max_msi = min(1000, msi_adata.shape[0])
                    
                    rna_idx = np.random.choice(rna_adata.shape[0], max_rna, replace=False)
                    msi_idx = np.random.choice(msi_adata.shape[0], max_msi, replace=False)
                    
                    rna_features = rna_adata.X[rna_idx, :50]
                    if hasattr(rna_features, 'toarray'):
                        rna_features = rna_features.toarray()
                    
                    msi_features = msi_adata.X[msi_idx, :50]
                    if hasattr(msi_features, 'toarray'):
                        msi_features = msi_features.toarray()
                    
                    scaler = RobustScaler()
                    rna_features = scaler.fit_transform(rna_features)
                    msi_features = scaler.fit_transform(msi_features)
                    
                    rna_feat_t = torch.tensor(rna_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                    rna_coord_t = torch.tensor(rna_coords_aligned[rna_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                    msi_feat_t = torch.tensor(msi_features, dtype=torch.float32, device=self.device).unsqueeze(0)
                    msi_coord_t = torch.tensor(msi_coords[msi_idx], dtype=torch.float32, device=self.device).unsqueeze(0)
                    
                    outputs = self.cross_modal_transformer(
                        rna_feat_t, rna_coord_t, msi_feat_t, msi_coord_t,
                        return_squeezed=False
                    )
                    
                    rna_emb = outputs['rna_embedding']
                    msi_emb = outputs['msi_embedding']
                    
                    loss = contrastive_loss(rna_emb, msi_emb)
                    sim_score = outputs['similarity_score']
                    alignment_loss = 1 - sim_score.mean()
                    loss = loss + 0.5 * alignment_loss
                    
                    epoch_val_loss += loss.item()
                    n_val_batches += 1
            
            val_losses.append(epoch_val_loss / max(n_val_batches, 1))
            
            # Save best model
            if val_losses[-1] < best_val_loss:
                best_val_loss = val_losses[-1]
                best_model_state = self.cross_modal_transformer.state_dict().copy()
            
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Train: {train_losses[-1]:.4f}, Val: {val_losses[-1]:.4f}")
        
        # Load best model
        if best_model_state is not None:
            self.cross_modal_transformer.load_state_dict(best_model_state)
            print(f"  Loaded best model with val loss: {best_val_loss:.4f}")
        
        # Plot train and validation curves
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss', color='blue')
        plt.plot(val_losses, label='Validation Loss', color='red', linestyle='--')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Cross-Modal Transformer Training')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Mark best validation point
        best_epoch = np.argmin(val_losses)
        plt.axvline(x=best_epoch, color='green', linestyle=':', alpha=0.7)
        plt.scatter([best_epoch], [val_losses[best_epoch]], color='green', s=100, zorder=5, label=f'Best Val (epoch {best_epoch})')
        
        plt.legend()
        plt.savefig(os.path.join(self.output_dir, 'training_curves', 'transformer_loss.png'), dpi=150)
        plt.close()
        
        print(f"  Final train loss: {train_losses[-1]:.4f}, Final val loss: {val_losses[-1]:.4f}")
    
    def extract_signature(
        self,
        coords: np.ndarray,
        values: np.ndarray,
        sample_id: str,
        feature_name: str,
        feature_type: str,
        msi_coords: Optional[np.ndarray] = None
    ) -> SpatialSignature:
        """Extract spatial signature"""
        
        aligned = None
        if feature_type == 'gene' and msi_coords is not None:
            aligned = align_rna_to_msi(coords, msi_coords)
        
        return SpatialSignature(
            sample_id=sample_id,
            feature_name=feature_name,
            feature_type=feature_type,
            transformer_embedding=None,
            coordinates=coords,
            raw_values=values,
            aligned_coordinates=aligned
        )
    
    def compute_transformer_embeddings(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature
    ) -> Tuple[np.ndarray, np.ndarray, dict]:
        """Compute Cross-Modal Transformer embeddings for a gene-mz pair"""
        
        if self.cross_modal_transformer is None:
            return None, None, {}
        
        self.cross_modal_transformer.eval()
        
        # Use aligned coordinates for gene
        gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
        mz_coords = mz_sig.coordinates
        
        # Prepare features
        gene_values = gene_sig.raw_values.reshape(-1, 1)
        mz_values = mz_sig.raw_values.reshape(-1, 1)
        
        scaler = RobustScaler()
        gene_scaled = scaler.fit_transform(gene_values)
        mz_scaled = scaler.fit_transform(mz_values)
        
        # Convert to tensors
        gene_feat_t = torch.tensor(gene_scaled, dtype=torch.float32, device=self.device).unsqueeze(0)
        gene_coord_t = torch.tensor(gene_coords, dtype=torch.float32, device=self.device).unsqueeze(0)
        mz_feat_t = torch.tensor(mz_scaled, dtype=torch.float32, device=self.device).unsqueeze(0)
        mz_coord_t = torch.tensor(mz_coords, dtype=torch.float32, device=self.device).unsqueeze(0)
        
        with torch.no_grad():
            outputs = self.cross_modal_transformer(gene_feat_t, gene_coord_t, mz_feat_t, mz_coord_t)
        
        attention_info = {
            'rna_to_msi': outputs['cross_attention_rna_to_msi'].cpu().numpy(),
            'msi_to_rna': outputs['cross_attention_msi_to_rna'].cpu().numpy(),
            'similarity': outputs['similarity_score'].cpu().numpy()
        }
        
        return (
            outputs['rna_embedding'].cpu().numpy(),
            outputs['msi_embedding'].cpu().numpy(),
            attention_info
        )
    
    def visualize_signature(self, sig: SpatialSignature, save_path: str):
        """Visualize gene/mz expression patterns"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # Row 0: Expression patterns
        # Original coordinates expression
        im1 = axes[0, 0].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.raw_values, cmap='viridis', s=10)
        axes[0, 0].set_title(f'{sig.feature_name} - Original', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0], label='Expression')
        
        # Aligned coordinates (for genes) or log-transformed
        if sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(sig.aligned_coordinates[:, 0], sig.aligned_coordinates[:, 1],
                                      c=sig.raw_values, cmap='viridis', s=10)
            axes[0, 1].set_title('Aligned (180° rotated)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='Expression')
        else:
            # Show log-transformed for better visualization
            log_vals = np.log1p(sig.raw_values)
            im2 = axes[0, 1].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                      c=log_vals, cmap='viridis', s=10)
            axes[0, 1].set_title('Log-transformed', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1], label='log(1+x)')
        axes[0, 1].set_xlabel('X (μm)')
        axes[0, 1].set_ylabel('Y (μm)')
        
        # Expression percentiles spatial view
        coords = sig.aligned_coordinates if sig.aligned_coordinates is not None else sig.coordinates
        thresh_90 = np.percentile(sig.raw_values, 90)
        mask_90 = sig.raw_values >= thresh_90
        axes[0, 2].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=5, alpha=0.3)
        if mask_90.sum() > 0:
            im3 = axes[0, 2].scatter(coords[mask_90, 0], coords[mask_90, 1],
                                      c=sig.raw_values[mask_90], cmap='hot', s=15)
            plt.colorbar(im3, ax=axes[0, 2], label='Expression')
        axes[0, 2].set_title(f'Top 10% Expression ({mask_90.sum()} spots)', fontweight='bold')
        axes[0, 2].set_xlabel('X (μm)')
        axes[0, 2].set_ylabel('Y (μm)')
        
        # Row 1: More visualizations
        # Expression histogram
        axes[1, 0].hist(sig.raw_values, bins=50, color='steelblue', edgecolor='darkblue', alpha=0.7)
        axes[1, 0].set_title('Expression Distribution', fontweight='bold')
        axes[1, 0].set_xlabel('Expression Value')
        axes[1, 0].set_ylabel('Count')
        
        # Expression percentiles visualization
        percentiles = [25, 50, 75, 90]
        colors = ['lightblue', 'steelblue', 'orange', 'red']
        axes[1, 1].scatter(coords[:, 0], coords[:, 1], c='lightgray', s=3, alpha=0.3)
        for pct, color in zip(percentiles, colors):
            thresh = np.percentile(sig.raw_values, pct)
            mask = sig.raw_values >= thresh
            axes[1, 1].scatter(coords[mask, 0], coords[mask, 1], 
                              c=color, s=5, alpha=0.5, label=f'≥{pct}th pct')
        axes[1, 1].set_title('Expression Percentiles', fontweight='bold')
        axes[1, 1].set_xlabel('X (μm)')
        axes[1, 1].set_ylabel('Y (μm)')
        axes[1, 1].legend(loc='upper right', fontsize=8)
        
        # Summary statistics
        stats_text = f"""
EXPRESSION STATISTICS
═══════════════════════════════

Feature: {sig.feature_name}
Sample:  {sig.sample_id}
Type:    {sig.feature_type.upper()}

Expression Values:
  Mean:   {sig.raw_values.mean():.4f}
  Std:    {sig.raw_values.std():.4f}
  Min:    {sig.raw_values.min():.4f}
  Max:    {sig.raw_values.max():.4f}
  Median: {np.median(sig.raw_values):.4f}

Spatial Metrics:
  Total Spots:   {len(sig.raw_values)}
  Non-zero:      {(sig.raw_values > 0).sum()}
  Sparsity:      {(sig.raw_values == 0).sum() / len(sig.raw_values) * 100:.1f}%
        """
        axes[1, 2].text(0.05, 0.95, stats_text, transform=axes[1, 2].transAxes,
                       fontsize=10, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))
        axes[1, 2].axis('off')
        
        plt.suptitle(f'EXPRESSION PATTERN: {sig.feature_name} ({sig.sample_id})',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_match(self, gene_sig, mz_sig, trans_sim, save_path):
        """Visualize match with transformer attention"""
        combined = compute_combined_score(trans_sim)
        
        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        
        # Row 0: Gene
        im1 = axes[0, 0].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        if gene_sig.aligned_coordinates is not None:
            im2 = axes[0, 1].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                                      c=gene_sig.raw_values, cmap='viridis', s=15)
            axes[0, 1].set_title('Gene (180° Aligned)', fontweight='bold')
            plt.colorbar(im2, ax=axes[0, 1])
        
        # Gene top expression
        gene_coords = gene_sig.aligned_coordinates if gene_sig.aligned_coordinates is not None else gene_sig.coordinates
        gene_thresh = np.percentile(gene_sig.raw_values, 90)
        gene_mask = gene_sig.raw_values >= gene_thresh
        axes[0, 2].scatter(gene_coords[:, 0], gene_coords[:, 1], c='lightgray', s=5, alpha=0.3)
        if gene_mask.sum() > 0:
            im3 = axes[0, 2].scatter(gene_coords[gene_mask, 0], gene_coords[gene_mask, 1],
                                      c=gene_sig.raw_values[gene_mask], cmap='hot', s=15)
            plt.colorbar(im3, ax=axes[0, 2])
        axes[0, 2].set_title('Gene Top 10% Expression', fontweight='bold')
        
        # Row 1: m/z
        im4 = axes[1, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.raw_values, cmap='viridis', s=5)
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 0])
        
        # m/z top expression
        mz_thresh = np.percentile(mz_sig.raw_values, 90)
        mz_mask = mz_sig.raw_values >= mz_thresh
        axes[1, 1].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1], c='lightgray', s=3, alpha=0.3)
        if mz_mask.sum() > 0:
            im5 = axes[1, 1].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                                      c=mz_sig.raw_values[mz_mask], cmap='hot', s=8)
            plt.colorbar(im5, ax=axes[1, 1])
        axes[1, 1].set_title('m/z Top 10% Expression', fontweight='bold')
        
        # Expression histograms comparison
        axes[1, 2].hist(gene_sig.raw_values, bins=30, alpha=0.5, label='Gene', color='blue')
        axes[1, 2].hist(mz_sig.raw_values, bins=30, alpha=0.5, label='m/z', color='red')
        axes[1, 2].set_title('Expression Distributions', fontweight='bold')
        axes[1, 2].legend()
        
        # Row 2: Overlays and metrics
        if gene_sig.aligned_coordinates is not None:
            axes[2, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                              c='blue', s=3, alpha=0.3, label='m/z')
            axes[2, 0].scatter(gene_sig.aligned_coordinates[:, 0], gene_sig.aligned_coordinates[:, 1],
                              c='red', s=10, alpha=0.5, label='Gene')
            axes[2, 0].set_title('Overlay (Gene aligned)', fontweight='bold')
            axes[2, 0].legend()
        
        # Top expression overlay
        if gene_sig.aligned_coordinates is not None:
            axes[2, 1].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                              c='blue', s=8, alpha=0.5, label='m/z top 10%')
            axes[2, 1].scatter(gene_coords[gene_mask, 0], gene_coords[gene_mask, 1],
                              c='red', s=15, alpha=0.7, label='Gene top 10%')
            axes[2, 1].set_title('Top 10% Overlay', fontweight='bold')
            axes[2, 1].legend()
        
        # Metrics with transformer scores
        trans_cosine = trans_sim.get('transformer_cosine', 0) if trans_sim else 0
        trans_l2 = trans_sim.get('transformer_l2_similarity', 0) if trans_sim else 0
        
        metrics_text = f"""
COMBINED SCORE: {combined:.4f}
═══════════════════════════════════

TRANSFORMER-BASED (100%):
  Transformer cosine:   {trans_cosine:.4f}
  Transformer L2 sim:   {trans_l2:.4f}
        """
        axes[2, 2].text(0.05, 0.95, metrics_text, transform=axes[2, 2].transAxes,
                       fontsize=9, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[2, 2].axis('off')
        
        plt.suptitle(f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name} | Score: {combined:.3f}',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def find_matches(self, gene_sig, mz_sigs, top_k=20):
        """Find matches using transformer similarity"""
        matches = []
        
        for mz_name, mz_sig in mz_sigs.items():
            # Compute transformer similarity
            if self.cross_modal_transformer is not None:
                gene_emb, mz_emb, attn_info = self.compute_transformer_embeddings(gene_sig, mz_sig)
                
                if gene_emb is not None:
                    gene_sig.transformer_embedding = gene_emb
                    mz_sig.transformer_embedding = mz_emb
                
                trans_sim = compute_transformer_similarity(gene_sig, mz_sig)
            else:
                trans_sim = {'transformer_cosine': 0, 'transformer_l2_similarity': 0}
            
            combined = compute_combined_score(trans_sim)
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                **trans_sim,
                'combined_score': combined
            })
        
        return pd.DataFrame(matches).sort_values('combined_score', ascending=False).head(top_k)
    
    def run_analysis(self, epochs_transformer=100, top_k=20):
        """Run analysis with Cross-Modal Transformer"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V10")
        print(f"RNA: {RNA_PIXEL_SIZE}μm (hexagonal) | MSI: {MSI_PIXEL_SIZE}μm (Cartesian)")
        print("Transformer-Only Matching")
        print("="*70)
        
        # Gene availability
        gene_avail = {}
        for gene in AAD_TARGET_GENES:
            gene_avail[gene] = {s: gene in self.rna_data[s].var_names 
                               for s in RNA_SAMPLE_IDS if s in self.rna_data}
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            n = sum(gene_avail[gene].values())
            print(f"  {gene}: {n}/{len(RNA_SAMPLE_IDS)}")
        
        # Prepare data for transformer training - use ALL samples
        print("\n" + "-"*70)
        print("PHASE 1: Preparing Data for Transformer")
        print("-"*70)
        
        rna_train = {}
        for sid in self.rna_data.keys():
            adata = self.rna_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            rna_train[sid] = {'coords': coords, 'features': features}
            print(f"  RNA {sid}: {coords.shape[0]} spots")
        
        msi_train = {}
        for sid in self.msi_data.keys():
            adata = self.msi_data[sid]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            features = adata.X[:, :50].toarray() if hasattr(adata.X, 'toarray') else adata.X[:, :50]
            msi_train[sid] = {'coords': coords, 'features': features}
            print(f"  MSI {sid}: {coords.shape[0]} spots")
        
        # Train Cross-Modal Transformer
        print("\n" + "-"*70)
        print("PHASE 2: Training Cross-Modal Transformer")
        print("-"*70)
        self.train_cross_modal_transformer(rna_train, msi_train, epochs_transformer)
        
        # Match
        print("\n" + "-"*70)
        print("PHASE 3: Matching (All Samples)")
        print("-"*70)
        
        all_results = []
        all_top5_results = []  # Store top 5 matches per gene-sample for detailed CSV     
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_avail[gene].items() if a]
            if not available:
                continue
            
            for rna_sample in available:
                print(f"\n  {rna_sample}")
                
                adata = self.rna_data[rna_sample]
                rna_coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                gene_idx = list(adata.var_names).index(gene)
                gene_expr = adata.X[:, gene_idx].toarray().flatten() if hasattr(adata.X, 'toarray') \
                           else adata.X[:, gene_idx].flatten()
                
                msi_sample = rna_sample
                if msi_sample not in self.msi_data:
                    print(f"    MSI {msi_sample} not found")
                    continue
                
                msi_adata = self.msi_data[msi_sample]
                msi_coords = np.column_stack([msi_adata.obs['x_um'].values, msi_adata.obs['y_um'].values])
                
                gene_sig = self.extract_signature(
                    rna_coords, gene_expr, rna_sample, gene, 'gene', msi_coords
                )
                
                # Gene visualization - focuses on expression patterns
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'gene_visualizations', f'{gene}_{rna_sample}.png')
                )

                print(f"    Matching vs MSI...")
                msi_data = msi_adata.X.toarray() if hasattr(msi_adata.X, 'toarray') else msi_adata.X
                mz_names = list(msi_adata.var_names)
                
                mz_sigs = {}
                for i, mz_name in enumerate(mz_names):
                    mz_sigs[mz_name] = self.extract_signature(
                        msi_coords, msi_data[:, i], msi_sample, mz_name, 'mz'
                    )
                    if (i + 1) % 100 == 0:
                        print(f"      {i+1}/{len(mz_names)}")
                
                matches = self.find_matches(gene_sig, mz_sigs, top_k)
                all_results.append(matches)


                # Store top 5 for the detailed CSV
                top5_matches = matches.head(5).copy()
                top5_matches['gene'] = gene
                top5_matches['rna_sample'] = rna_sample
                top5_matches['rank'] = range(1, len(top5_matches) + 1)
                all_top5_results.append(top5_matches)
                
                if len(matches) > 0:
                    print(f"\n    Top 5:")
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        trans_score = m.get('transformer_cosine', 0)
                        print(f"      {idx+1}. m/z {m['mz_feature']}: {m['combined_score']:.3f} (trans: {trans_score:.3f})")
                    
                    # Visualize top match        
                    top = matches.iloc[0]
                    top_mz = mz_sigs[top['mz_feature']]
                    trans_sim = compute_transformer_similarity(gene_sig, top_mz)
                    
                    self.visualize_match(
                        gene_sig, top_mz, trans_sim,
                        os.path.join(self.output_dir, 'match_visualizations', f'{gene}_{rna_sample}_top.png')
                    )
        
        # Save results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            results.to_csv(os.path.join(self.output_dir, 'gene_to_mz_matches_v10.csv'), index=False)
             
            # Save top 5 matches per gene-sample with all scores
            if all_top5_results:
                top5_df = pd.concat(all_top5_results, ignore_index=True)
                
                # Reorder columns for better readability
                priority_cols = ['gene', 'rna_sample', 'rank', 'mz_feature', 'msi_sample', 'combined_score']
                other_cols = [c for c in top5_df.columns if c not in priority_cols]
                top5_df = top5_df[priority_cols + other_cols]
                
                # Sort by gene, sample, then rank
                top5_df = top5_df.sort_values(['gene', 'rna_sample', 'rank'])
                
                top5_df.to_csv(os.path.join(self.output_dir, 'gene_to_mz_top5_matches_all_scores.csv'), index=False)
                print(f"\nSaved top 5 matches per gene-sample to: gene_to_mz_top5_matches_all_scores.csv")
                print(f"  Total entries: {len(top5_df)}")
                print(f"  Genes: {top5_df['gene'].nunique()}")
                print(f"  Samples: {top5_df['rna_sample'].nunique()}")
            
            print("\n" + "="*70)
            print("TOP MATCHES")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gr = results[results['gene'] == gene]
                if len(gr) > 0:
                    t = gr.iloc[0]
                    trans_score = t.get('transformer_cosine', 0)
                    print(f"\n{gene}: m/z {t['mz_feature']} ({t['rna_sample']}) = {t['combined_score']:.3f} (trans: {trans_score:.3f})")
            
            return results
        
        return None
    


def main():
    print("="*70)
    print("V10: Transformer-Only Matching")
    print(f"RNA: {RNA_PIXEL_SIZE}μm | MSI: {MSI_PIXEL_SIZE}μm")
    print("="*70)
    
    matcher = HybridTransformerMatcher(output_dir='./gene_to_mz_results_v10')
    matcher.load_all_data()
    results = matcher.run_analysis(epochs_transformer=100, top_k=20)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    
    return matcher, results


if __name__ == "__main__":
    matcher, results = main()